In [ ]:
from pathlib import Path
import zipfile
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

%pip install openpyxl


In [ ]:

zip_path = "Pistachio_Image_Dataset.zip"
xls_name = "Pistachio_Image_Dataset/Pistachio_16_Features_Dataset/Pistachio_16_Features_Dataset.xlsx"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open(xls_name) as f:
        df = pd.read_excel(f)

df.head()

In [ ]:
numeric_features = [
    "AREA",
    "PERIMETER",
    "MAJOR_AXIS",
    "MINOR_AXIS",
    "ECCENTRICITY",
    "EQDIASQ",
    "SOLIDITY",
    "CONVEX_AREA",
    "EXTENT",
    "ASPECT_RATIO",
    "ROUNDNESS",
    "COMPACTNESS",
    "SHAPEFACTOR_1",
    "SHAPEFACTOR_2",
    "SHAPEFACTOR_3",
    "SHAPEFACTOR_4"
]
# 将最后一列 Class 转为 0/1
df['Class'] = df['Class'].astype(str).str.strip()

mapping = {
    'Kirmizi_Pistachio': 0,
    'Siit_Pistachio': 1,
}

df['Class'] = df['Class'].map(mapping).astype(int)

X = df[numeric_features]
y = df['Class']

In [ ]:
print("数据形状:", df.shape)
print("缺失值数量:", int(df.isna().sum().sum()))
print("\n目标变量分布:")
print(y.value_counts())
y = y.replace(["Kirmizi_Pistachio", "Siit_Pistachio"], [0, 1])
print(y.dtype)
print(f"\n故障比例: {y.mean():.2%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("训练集:", X_train.shape)
print("测试集:", X_test.shape)
print("训练集故障比例:", f"{y_train.mean():.2%}")
print("测试集故障比例:", f"{y_test.mean():.2%}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
    ]
)

In [ ]:
knn = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=5, weights="uniform")),
    ]
)

knn.fit(X_train, y_train)

In [ ]:
y_pred = knn.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Kirmizi_Pistachio", "Siit_Pistachio"], digits=3))

In [ ]:
k_values = [1, 3, 5, 7, 9, 15, 21, 31]
rows = []

for k in k_values:
    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", KNeighborsClassifier(n_neighbors=k, weights="uniform")),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rows.append(
        {
            "k": k,
            "accuracy": model.score(X_test, y_test),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred, zero_division=0),
        }
    )

k_results = pd.DataFrame(rows)
k_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_results["k"], k_results["precision"], marker="o", label="Precision")
plt.plot(k_results["k"], k_results["recall"], marker="o", label="Recall")
plt.plot(k_results["k"], k_results["f1"], marker="o", label="F1")
plt.xlabel("K")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("不同 K 值下的测试集表现")
plt.show()

## 超参数

In [ ]:
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
import xgboost as xgb
import lightgbm as lgb


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
    ]
)

base_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            lgb.LGBMClassifier(
                n_estimators=200,
                objective="binary",
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
                force_col_wise=True,
                verbosity=-1,
            ),
        ),
    ]
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

In [ ]:
param_grid = {
    "model__num_leaves": [7, 15, 31],
    "model__learning_rate": [0.05, 0.1],
    "model__min_child_samples": [10, 20],
}

grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    verbose=0,
)

grid_search.fit(X_train, y_train)

In [ ]:
print("Grid Search 最佳参数:")
print(grid_search.best_params_)
print(f"\nGrid Search 最佳交叉验证 F1: {grid_search.best_score_:.4f}")

In [ ]:
param_columns = [
    "param_model__num_leaves",
    "param_model__learning_rate",
    "param_model__min_child_samples",
    "param_model__subsample",
    "param_model__colsample_bytree",
]
result_columns = [
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]

grid_cv_results = pd.DataFrame(grid_search.cv_results_)

grid_param_columns = [c for c in param_columns if c in grid_cv_results.columns]

grid_top = grid_cv_results[grid_param_columns + result_columns].sort_values(
    "rank_test_score"
)

print("Grid Search 前 10 组参数：")
display(grid_top.head(10).round(4))

In [ ]:
import seaborn as sns

grid_heatmap_data = grid_cv_results.pivot_table(
    index="param_model__num_leaves",
    columns="param_model__learning_rate",
    values="mean_test_score",
    aggfunc="mean",
)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    grid_heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    ax=axes[0],
)
axes[0].set_title("Grid Search：平均交叉验证 F1")
axes[0].set_xlabel("learning_rate")
axes[0].set_ylabel("num_leaves")

plt.tight_layout()
plt.show()